# pyTOST basic usage demo

This notebook shows the core `run_tost(...)` workflow on one synthetic paired-difference dataset. It keeps the focus on the mechanics of running each engine and comparing the primary results.

## 1) Imports and setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyTOST").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pyTOST import run_tost, WorkflowOptions, SpatialConfig, SpatioTemporalConfig
from pyTOST.data_gen.params_io import load_params, kwargs_for
from pyTOST.data_gen.synthetic_tost_data import generate_spatiotemporal

## 2) Build one synthetic dataset

We use a saved spatiotemporal parameter set so that every engine sees the same paired-difference table.

In [ ]:
def to_diff_df(df_long: pd.DataFrame) -> pd.DataFrame:
    index_cols = ["x", "y_sp", "t", "sample_id"]

    wide = (
        df_long[index_cols + ["arm", "y"]]
        .pivot_table(index=index_cols, columns="arm", values="y", aggfunc="first")
        .reset_index()
    )
    wide.columns.name = None
    wide = wide.rename(columns={"y_sp": "ycoord", "t": "time"})

    # Build a stable spatial location id shared across time slices.
    wide["cluster_id"] = wide.groupby(["x", "ycoord"]).ngroup().astype(int)
    wide["diff"] = wide["B"] - wide["A"]
    return wide[["sample_id", "cluster_id", "x", "ycoord", "time", "diff"]]

params = load_params(REPO_ROOT / "pyTOST" / "data_gen" / "best_spatiotemporal_params.json")
df_long, _ = generate_spatiotemporal(**kwargs_for(generate_spatiotemporal, params))
df = to_diff_df(df_long)
df.head()

## 3) Run the core workflow for each engine

In [ ]:
COMMON = dict(y="diff", margins=[1.0], alpha=0.05, options=WorkflowOptions(do_sensitivity=False, bootstrap_B=25, seed=42))

res_iid = run_tost(df, engine="iid", **COMMON)
res_cluster = run_tost(df, engine="cluster", cluster="cluster_id", **COMMON)
res_temporal = run_tost(df, engine="temporal", time="time", **COMMON)
res_spatial = run_tost(
    df,
    engine="spatial",
    cluster="cluster_id",
    x="x",
    ycoord="ycoord",
    spatial_config=SpatialConfig(nu_grid=(0.5, 1.5, 2.5), per_cluster_nugget=True, verbose_diagnostics=False),
    **COMMON,
)
res_spatiotemporal = run_tost(
    df,
    engine="spatiotemporal",
    cluster="cluster_id",
    time="time",
    x="x",
    ycoord="ycoord",
    spatiotemporal_config=SpatioTemporalConfig(
        nu_grid=(0.5, 1.5, 2.5),
        per_cluster_nugget=True,
        verbose_diagnostics=False,
        mu_bootstrap_B=40,
        mu_bootstrap_seed=42,
    ),
    **COMMON,
)

## 4) Compare the primary results

In [ ]:
def stack_primary(name, res):
    out = res["primary"].copy()
    out.insert(0, "engine", name)
    return out

summary = pd.concat([
    stack_primary("IID", res_iid),
    stack_primary("Cluster", res_cluster),
    stack_primary("Temporal", res_temporal),
    stack_primary("Spatial", res_spatial),
    stack_primary("Spatiotemporal", res_spatiotemporal),
], ignore_index=True)
summary[["engine", "delta", "mu_hat", "ci_low", "ci_high", "equivalent", "method"]]

## 5) Takeaway

The basic workflow is the same for every engine: provide the paired-difference column, choose an engine, add the columns required by that engine, and inspect the `primary` result table. The advanced notebook expands this by showing validation, interpretation, and alternative spatiotemporal configurations.